# ADIP — Phase 3: Data Understanding & Exploratory Data Analysis (EDA)
**Author:** Anshika (AI/ML & Data Intelligence Lead)  
**Project:** AI Architecture Decision Intelligence Platform (ADIP)  
**Track:** Phase 3 Deliverable — Data Strategy, Dataset Schemas & Exploratory Feature Modeling

---

### Executive Overview & Purpose
In software engineering, architectural decisions (e.g., choosing between Monolith vs. Microservices, selecting relational vs. NoSQL datastores, adopting event queues, or provisioning horizontal Kubernetes replicas) are traditionally driven by intuition, dogma, or post-incident retrospectives. **ADIP** replaces subjective guesswork with data-driven architecture intelligence.

To train high-accuracy machine learning surrogate models for multi-objective architecture evaluation, we must bridge the gap between **high-level requirements**, **structural graph topologies**, and **low-level empirical operational metrics** (latency percentiles, throughput ceilings, financial cloud expenditure, and failure rates).

### Objectives:
1. **Source Landscape & Evidence Types:** Audit the 15 curated data sources across 5 evidence types from `ml/configs/data.yaml`.
2. **Schema & Provenance Verification:** Validate the 5 linked datasets (Datasets A–E) and immutable provenance audit trails.
3. **Benchmark Data Ingestion & Synthesis:** Load and synthesize multi-tier benchmark data adhering strictly to `ml/configs/features.yaml`.
4. **Empirical Distribution & Tail Latency Analysis:** Characterize non-linear queuing bottlenecks, evaluating $p_{50}$ vs. $p_{95}$ vs. $p_{99}$ latency distributions across architectures.
5. **Graph Topology Extraction:** Leverage `networkx` and `ArchitectureFeatureBuilder` to extract structural metrics (`graph_depth`, `fan-in`, `fan-out`, `complexity_score`).
6. **Handshake to Phase 4:** Perform preliminary sanity checks, outlier inspections, and target leakage identification.


## 1. Setup & Environment Configuration
We load standard numerical computing, graph analysis, and visualization libraries. We set global visualization styles for high-fidelity technical reporting.


In [ ]:
import os
import sys
from pathlib import Path
import yaml
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Add repository root to path for local module imports
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
if str(repo_root.parent) not in sys.path:
    sys.path.append(str(repo_root.parent))

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print(f"Working Directory: {repo_root}")
print("Environment configured successfully.")


## 2. Source Landscape & Evidence Taxonomy Audit
ADIP incorporates **15 heterogeneous data sources** classified into **5 distinct Evidence Types**. 
Per our Phase 3 data strategy, mixing synthetic data with real telemetry without explicit provenance tracking corrupts surrogate models. We inspect `ml/configs/data.yaml` to verify the source catalog and metadata requirements.


In [ ]:
data_config_path = Path("ml/configs/data.yaml")
if not data_config_path.exists():
    data_config_path = Path("../configs/data.yaml")

with open(data_config_path, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

sources = data_config.get("data_sources", {})
print(f"Total Configured Data Sources: {len(sources)}")

# Tabulate sources and evidence taxonomy
source_records = []
for k, v in sources.items():
    source_records.append({
        "Source Key": k,
        "Source Name": v.get("name"),
        "Evidence Type": v.get("evidence_type"),
        "Priority": v.get("priority"),
        "Is Real": v.get("is_real"),
        "Is Benchmark": v.get("is_benchmark"),
        "Is Simulated": v.get("is_simulated"),
        "Is Synthetic": v.get("is_synthetic"),
        "License": v.get("license")
    })

df_sources = pd.DataFrame(source_records)
df_sources.head(10)


In [ ]:
# Visualizing Evidence Type Distribution and Data Fidelity
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
evidence_counts = df_sources['Evidence Type'].value_counts()
colors = ['#2b5c8f', '#4682b4', '#5f9ea0', '#e08244', '#708090']
evidence_counts.plot(kind='barh', color=colors[:len(evidence_counts)], edgecolor='black')
plt.title("ADIP Data Sources by Evidence Type", fontweight='bold')
plt.xlabel("Number of Data Sources")
plt.ylabel("")

plt.subplot(1, 2, 2)
fidelity_matrix = df_sources[['Is Real', 'Is Benchmark', 'Is Simulated', 'Is Synthetic']].sum()
fidelity_matrix.plot(kind='bar', color=['#2e7d32', '#1976d2', '#f57c00', '#d32f2f'], edgecolor='black')
plt.title("Data Fidelity Classification", fontweight='bold')
plt.ylabel("Number of Sources")
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()


## 3. Schema Exploration: The 5 Linked Datasets
In Phase 3, we architected the system into **5 Normalized, Linked Datasets (Datasets A–E)** linked by composite foreign keys (`project_id`, `architecture_id`, `workload_id`, `run_id`).
This design prevents data duplication, supports modular feature engineering, and enables flexible join strategies.


In [ ]:
features_config_path = Path("ml/configs/features.yaml")
if not features_config_path.exists():
    features_config_path = Path("../configs/features.yaml")

with open(features_config_path, "r", encoding="utf-8") as f:
    features_config = yaml.safe_load(f)

datasets = features_config.get("datasets", {})
print("Registered Linked Datasets:")
for d_key, d_val in datasets.items():
    pk = d_val.get("primary_key", "None")
    fields = list(d_val.get("fields", {}).keys())
    print(f"  * {d_key.upper()} (PK: {pk}) -> {len(fields)} fields")
    print(f"    Sample fields: {', '.join(fields[:6])}...")


### Provenance Tracking Contract
Every ingested row must satisfy the **Provenance Schema** from `ml/configs/data.yaml` to ensure complete reproducibility. If a metric originates from a synthetic benchmark rather than real operational telemetry, downstream surrogate models can apply appropriate confidence weighting.


In [ ]:
prov_fields = data_config.get("provenance_schema", {}).get("mandatory_fields", [])
print("Mandatory Provenance Fields:")
for idx, field in enumerate(prov_fields, 1):
    print(f"  {idx}. {field}")


## 4. Benchmark Dataset Ingestion & Simulation
To explore cross-feature dynamics prior to complete physical cluster execution, we generate an empirical benchmark dataset strictly conforming to `ml/configs/features.yaml`.
The dataset models realistic queuing behaviors, network latencies, cache hit/miss dynamics, and cloud pricing formulas.


In [ ]:
def generate_sample_benchmark_dataset(n_samples: int = 1200, random_state: int = 42) -> pd.DataFrame:
    """
    Generates a realistic multi-architecture benchmark dataset conforming to features.yaml bounds.
    Models queuing delay, cache effects, service fan-out, and cloud cost.
    """
    np.random.seed(random_state)
    
    # Categoricals
    arch_patterns = ["monolith", "modular_monolith", "microservices", "event_driven", "serverless"]
    databases = ["postgres", "mysql", "mongodb", "dynamodb", "cassandra"]
    caches = ["none", "redis", "memcached", "hazelcast"]
    messagings = ["none", "kafka", "rabbitmq", "aws_sqs"]
    deployments = ["kubernetes", "aws_ecs", "docker_compose", "aws_lambda"]
    clouds = ["aws", "gcp", "azure"]

    pattern_choices = np.random.choice(arch_patterns, size=n_samples, p=[0.20, 0.20, 0.35, 0.15, 0.10])
    db_choices = np.random.choice(databases, size=n_samples)
    cache_choices = np.random.choice(caches, size=n_samples, p=[0.25, 0.45, 0.20, 0.10])
    msg_choices = np.random.choice(messagings, size=n_samples, p=[0.30, 0.40, 0.20, 0.10])
    deploy_choices = np.random.choice(deployments, size=n_samples)
    cloud_choices = np.random.choice(clouds, size=n_samples, p=[0.55, 0.25, 0.20])

    records = []
    for i in range(n_samples):
        pattern = pattern_choices[i]
        db = db_choices[i]
        cache = cache_choices[i]
        msg = msg_choices[i]
        deploy = deploy_choices[i]
        cloud = cloud_choices[i]

        # Sizing and topology based on pattern
        if pattern == "monolith":
            service_count = 1
            node_count = 1
            edge_count = 0
            graph_depth = 1
            max_fan_in = 0
            max_fan_out = 0
            replica_count = int(np.random.choice([1, 2, 4, 8]))
            cpu_limit = replica_count * np.random.choice([4.0, 8.0, 16.0])
            memory_limit = replica_count * np.random.choice([8.0, 16.0, 32.0])
        elif pattern == "modular_monolith":
            service_count = int(np.random.randint(3, 7))
            node_count = service_count
            edge_count = service_count - 1
            graph_depth = 2
            max_fan_in = 1
            max_fan_out = 2
            replica_count = int(np.random.choice([2, 4, 8]))
            cpu_limit = replica_count * np.random.choice([4.0, 8.0])
            memory_limit = replica_count * np.random.choice([8.0, 16.0])
        elif pattern == "microservices":
            service_count = int(np.random.randint(6, 30))
            node_count = service_count
            edge_count = int(service_count * np.random.uniform(1.2, 2.8))
            graph_depth = int(np.random.randint(3, 7))
            max_fan_in = int(np.random.randint(2, 6))
            max_fan_out = int(np.random.randint(3, 8))
            replica_count = int(service_count * np.random.choice([2, 3, 5]))
            cpu_limit = replica_count * 1.0
            memory_limit = replica_count * 2.0
        elif pattern == "event_driven":
            service_count = int(np.random.randint(5, 20))
            node_count = service_count
            edge_count = int(service_count * np.random.uniform(1.0, 2.0))
            graph_depth = int(np.random.randint(2, 5))
            max_fan_in = int(np.random.randint(1, 4))
            max_fan_out = int(np.random.randint(2, 5))
            replica_count = int(service_count * 2)
            cpu_limit = replica_count * 1.5
            memory_limit = replica_count * 3.0
        else: # serverless
            service_count = int(np.random.randint(10, 40))
            node_count = service_count
            edge_count = int(service_count * 1.1)
            graph_depth = int(np.random.randint(2, 4))
            max_fan_in = 2
            max_fan_out = 3
            replica_count = int(np.random.randint(10, 100))
            cpu_limit = 0.5 * replica_count
            memory_limit = 1.0 * replica_count

        avg_degree = float(edge_count) / float(node_count) if node_count > 0 else 0.0
        complexity_score = float(node_count * 0.4 + edge_count * 0.6 + graph_depth * 1.5)

        # Workload metrics
        rps = float(np.random.exponential(scale=1500) + 50)
        concurrency = int(np.clip(rps * np.random.uniform(0.05, 0.25), 10, 50000))
        read_write_ratio = float(np.random.uniform(0.5, 0.95))
        data_size = float(np.random.uniform(5.0, 2500.0))
        request_size = float(np.random.uniform(0.5, 15.0))

        # Queuing load factor
        capacity = max(replica_count * 250.0, 100.0)
        load_factor = rps / capacity

        # Base network & processing latency
        base_proc_time = 8.0 + (graph_depth * 4.2)
        if cache != "none":
            cache_factor = 0.45 if read_write_ratio > 0.8 else 0.70
        else:
            cache_factor = 1.15
        
        # Latency percentiles with queuing explosion (strictly enforcing p50 <= p95 <= p99)
        p50 = max(float(base_proc_time * cache_factor + (load_factor ** 1.8) * 12.0 + np.random.uniform(-0.5, 1.5)), 2.0)
        p95 = max(float(p50 * (1.3 + 0.08 * max_fan_out) + (load_factor ** 2.2) * 25.0 + np.random.uniform(1.0, 4.0)), p50 + 1.0)
        p99 = max(float(p95 * (1.4 + 0.12 * graph_depth) + (load_factor ** 2.6) * 55.0 + np.random.uniform(2.0, 8.0)), p95 + 2.0)

        # Throughput
        throughput = min(rps, capacity * (1.0 + 0.15 * np.log1p(replica_count))) * np.random.uniform(0.92, 1.0)
        
        # Cost modeling
        v_cpu_rate = 32.0 if cloud == "aws" else (30.0 if cloud == "gcp" else 33.0)
        gb_mem_rate = 4.20
        compute_cost = (cpu_limit * v_cpu_rate) + (memory_limit * gb_mem_rate)
        storage_cost = data_size * 0.08
        network_cost = (rps * request_size * 2.6 * 0.000001) * 3600 * 24 * 30 * 0.09
        managed_services_cost = 0.0
        if cache != "none":
            managed_services_cost += 120.0
        if msg != "none":
            managed_services_cost += 180.0
        total_monthly_cost = compute_cost + storage_cost + network_cost + managed_services_cost

        # Ratios
        rps_per_service = rps / service_count
        concurrency_per_replica = concurrency / replica_count

        records.append({
            "architecture_id": f"arch_{i:04d}",
            "architecture_pattern": pattern,
            "database": db,
            "cache": cache,
            "messaging": msg,
            "deployment": deploy,
            "cloud": cloud,
            "service_count": service_count,
            "node_count": node_count,
            "edge_count": edge_count,
            "replica_count": replica_count,
            "cpu_limit": cpu_limit,
            "memory_limit": memory_limit,
            "graph_depth": graph_depth,
            "average_degree": avg_degree,
            "max_fan_in": max_fan_in,
            "max_fan_out": max_fan_out,
            "architecture_complexity_score": complexity_score,
            "rps": rps,
            "concurrency": concurrency,
            "read_write_ratio": read_write_ratio,
            "data_size": data_size,
            "request_size": request_size,
            "rps_per_service": rps_per_service,
            "concurrency_per_replica": concurrency_per_replica,
            "p50": round(p50, 2),
            "p95": round(p95, 2),
            "p99": round(p99, 2),
            "throughput": round(throughput, 1),
            "total_monthly_cost": round(total_monthly_cost, 2),
            "evidence_type": "Type_E_Controlled_Simulation"
        })

    return pd.DataFrame(records)

df = generate_sample_benchmark_dataset(n_samples=1500)
print(f"Generated Benchmark Dataset Shape: {df.shape}")
df.head()


## 5. Statistical Exploratory Data Analysis (EDA)

### 5.1 Tail Latency Analysis: $p_{50}$ vs. $p_{95}$ vs. $p_{99}$
A cornerstone principle in modern distributed systems (such as Dean & Barroso's *The Tail at Scale*) is that average or median ($p_{50}$) latency hides catastrophic tail latency ($p_{99}$).
When a user request fans out across $k$ services in parallel, the overall latency is dominated by the maximum response time of the slowest dependency:
$$P(\text{service latency} > T) = 1 - (1 - p)^k$$
Below, we evaluate how tail latencies amplify across different architecture patterns.


In [ ]:
plt.figure(figsize=(14, 5))

# Plot 50th vs 99th percentile across architecture patterns
melted_latencies = pd.melt(
    df,
    id_vars=['architecture_pattern'],
    value_vars=['p50', 'p95', 'p99'],
    var_name='Percentile',
    value_name='Latency_ms'
)

sns.boxplot(
    data=melted_latencies,
    x='architecture_pattern',
    y='Latency_ms',
    hue='Percentile',
    palette={'p50': '#4caf50', 'p95': '#ff9800', 'p99': '#e91e63'},
    showfliers=False
)

plt.title("Latency Percentile Distribution ($p_{50}, p_{95}, p_{99}$) by Architecture Pattern", fontweight='bold')
plt.xlabel("Architecture Pattern")
plt.ylabel("Response Time (ms)")
plt.yscale('log')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(title="Percentile", frameon=True)
plt.tight_layout()
plt.show()


### 5.2 Workload Intensity vs. Throughput Saturation (The Knee of the Curve)
Systems process requests linearly up to their saturation knee, after which concurrency contention, thread starvation, and GC pauses cause queuing latency to explode while throughput plateaus.


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(
    data=df,
    x='rps',
    y='throughput',
    hue='architecture_pattern',
    alpha=0.6,
    palette='tab10'
)
plt.plot([0, 6000], [0, 6000], 'r--', label='1:1 Ideal Capacity Line')
plt.title("Workload RPS vs Achieved Throughput", fontweight='bold')
plt.xlabel("Incoming Request Rate (RPS)")
plt.ylabel("Observed Throughput (req/s)")
plt.legend(title="Pattern", loc="upper left")

plt.subplot(1, 2, 2)
sns.scatterplot(
    data=df,
    x='rps_per_service',
    y='p99',
    hue='cache',
    alpha=0.7,
    palette='viridis'
)
plt.yscale('log')
plt.title("Workload Intensity (RPS/Service) vs $p_{99}$ Latency", fontweight='bold')
plt.xlabel("RPS per Microservice")
plt.ylabel("$p_{99}$ Latency (ms) [Log Scale]")
plt.legend(title="Cache Layer")

plt.tight_layout()
plt.show()


### 5.3 Cache Strategy & Read/Write Intensity
We observe the empirical impact of caching layers (`redis`, `memcached`, `hazelcast` vs. `none`) across varying database technologies and read-heavy workloads.


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.barplot(
    data=df,
    x='database',
    y='p95',
    hue='cache',
    errorbar=None,
    palette='Blues_d'
)
plt.title("$p_{95}$ Latency by Database & Cache Strategy", fontweight='bold')
plt.xlabel("Database Technology")
plt.ylabel("Mean $p_{95}$ Latency (ms)")
plt.legend(title="Cache Type", frameon=True)

plt.subplot(1, 2, 2)
sns.scatterplot(
    data=df,
    x='read_write_ratio',
    y='p50',
    hue='cache',
    palette='Set1',
    alpha=0.6
)
plt.title("Read/Write Ratio vs Median Latency ($p_{50}$)", fontweight='bold')
plt.xlabel("Read/Write Ratio (1.0 = 100% Reads)")
plt.ylabel("$p_{50}$ Latency (ms)")

plt.tight_layout()
plt.show()


### 5.4 Financial Cloud Cost Analysis Across Providers & Deployments
Architectural selection is inherently constrained by financial budgets. Here we evaluate total monthly cloud spend across AWS, GCP, Azure, and deployment platforms.


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.boxplot(
    data=df,
    x='deployment',
    y='total_monthly_cost',
    hue='cloud',
    palette='Set2',
    showfliers=False
)
plt.title("Monthly Cloud Infrastructure Cost by Deployment & Cloud", fontweight='bold')
plt.xlabel("Deployment Platform")
plt.ylabel("Total Monthly Cost (USD)")
plt.legend(title="Cloud Provider")

plt.subplot(1, 2, 2)
# Cost Efficiency: USD per 100k requests
df['cost_per_100k_reqs'] = (df['total_monthly_cost'] / (df['throughput'] * 3600 * 24 * 30)) * 100000
sns.boxplot(
    data=df,
    x='architecture_pattern',
    y='cost_per_100k_reqs',
    hue='architecture_pattern',
    palette='pastel',
    legend=False,
    showfliers=False
)
plt.title("Cost Efficiency ($ per 100,000 Served Requests)", fontweight='bold')
plt.xlabel("Architecture Pattern")
plt.ylabel("Cost per 100k Requests (USD)")

plt.tight_layout()
plt.show()


### 5.5 Correlation Analysis: Structural Features vs. Targets
We compute the Spearman rank-order correlation matrix to inspect non-linear relationships between graph topology features, workload features, and prediction targets.


In [ ]:
analysis_cols = [
    'service_count', 'replica_count', 'cpu_limit', 'memory_limit',
    'graph_depth', 'average_degree', 'max_fan_out', 'architecture_complexity_score',
    'rps', 'concurrency', 'p50', 'p95', 'p99', 'throughput', 'total_monthly_cost'
]

corr = df[analysis_cols].corr(method='spearman')

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    cmap='vlag',
    vmax=1.0,
    vmin=-1.0,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=.5,
    cbar_kws={"shrink": .8}
)
plt.title("Spearman Rank Correlation: Architecture, Workload & Operational Targets", fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Architecture Graph Topology & NetworkX Feature Extraction
In software architecture, systems are naturally represented as **directed service dependency graphs**:
- **Nodes ($V$)**: Individual services, databases, queues, and caches.
- **Edges ($E$)**: Network interactions (synchronous REST/gRPC or asynchronous messaging).

We utilize `ml.src.features.build_features.ArchitectureFeatureBuilder` to extract structural metrics:
- `graph_depth`: Maximum depth of synchronous call chains.
- `max_fan_in`: Highest number of incoming dependencies on a single service (bottleneck indicator).
- `max_fan_out`: Highest number of downstream dependencies invoked by a service (blast radius indicator).
- `architecture_complexity_score`: Aggregate structural complexity.


In [ ]:
# Instantiate the Phase 3 Feature Builder
from ml.src.features.build_features import ArchitectureFeatureBuilder

builder = ArchitectureFeatureBuilder()

# Construct a realistic Microservices Dependency Graph
G = nx.DiGraph()

# Add nodes
services = [
    "api_gateway", "auth_service", "user_service", "catalog_service",
    "order_service", "payment_service", "inventory_service",
    "notification_service", "postgres_db", "redis_cache", "kafka_bus"
]
G.add_nodes_from(services)

# Add directed dependency edges
edges = [
    ("api_gateway", "auth_service"),
    ("api_gateway", "catalog_service"),
    ("api_gateway", "order_service"),
    ("auth_service", "user_service"),
    ("auth_service", "redis_cache"),
    ("user_service", "postgres_db"),
    ("catalog_service", "redis_cache"),
    ("catalog_service", "postgres_db"),
    ("order_service", "inventory_service"),
    ("order_service", "payment_service"),
    ("order_service", "kafka_bus"),
    ("inventory_service", "postgres_db"),
    ("payment_service", "kafka_bus"),
    ("kafka_bus", "notification_service"),
]
G.add_edges_from(edges)

# Extract topological metrics
metrics = builder.compute_graph_metrics_from_networkx(G)
print("Extracted Graph Topology Metrics for Architecture:")
for k, v in metrics.items():
    print(f"  * {k}: {v}")


In [ ]:
# Visualizing the Architectural Dependency DAG
plt.figure(figsize=(12, 7))

# Calculate node degrees for visual weight
node_sizes = [1800 + 400 * (G.in_degree(n) + G.out_degree(n)) for n in G.nodes()]

# Layout
try:
    pos = nx.spring_layout(G, seed=42, k=1.2)
except Exception:
    pos = nx.circular_layout(G)

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='#3498db', edgecolors='#2c3e50', linewidths=2)
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, edge_color='#7f8c8d', width=2)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', font_color='white')

plt.title("Sample Microservice Dependency Graph ($N=11$ Nodes, $E=14$ Edges)", fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


## 7. Handshake to Phase 4: Data Quality & Leakage Prevention

### 7.1 Physical Sanity Bounds Audit
Before any data moves to model training, it must pass physical range validation. In Phase 3, we defined these invariants in `ml/configs/features.yaml`:
- **Read/Write Ratio**: Must be strictly bounded within $[0.0, 1.0]$.
- **CPU/Memory Utilization**: Must never exceed $100\%$.
- **Latency**: Must be strictly non-negative ($\ge 0$ ms).
- **Percentile Order Invariant**: $p_{50} \le p_{95} \le p_{99}$ must hold for every single observation.


In [ ]:
# Audit physical invariants on our benchmark data
p_order_violations = df[~((df['p50'] <= df['p95']) & (df['p95'] <= df['p99']))]
rw_violations = df[~((df['read_write_ratio'] >= 0.0) & (df['read_write_ratio'] <= 1.0))]
cost_violations = df[df['total_monthly_cost'] < 0.0]

print(f"Percentile Order Invariant Violations: {len(p_order_violations)}")
print(f"Read/Write Ratio Violations:          {len(rw_violations)}")
print(f"Negative Cost Violations:               {len(cost_violations)}")

assert len(p_order_violations) == 0, "Data quality violation detected in latency ordering!"
print("Physical sanity bounds verified successfully.")


### 7.2 Target Leakage Prevention
A critical risk identified in the Phase 4 requirements is **Target Leakage**:
Operational metrics (such as observed $p_{99}$ latency, actual throughput, and incurred cloud bills) are the **targets** our surrogate models predict. 

> **CRITICAL RULE**: When evaluating candidate architectures for a new software project, operational targets (`p50`, `p95`, `p99`, `throughput`, `total_monthly_cost`) are unknown. Therefore, they must **NEVER** be used as input features in recommendation or sizing pipelines.

Furthermore, when partitioning datasets into Train/Val/Test splits, random row-level splitting causes **Architectural Contamination** (the same architecture topology appearing in both train and test sets under slightly different RPS). As established in `ml/src/data/split.py`, we must partition using `GroupShuffleSplit` on `architecture_id`.


In [ ]:
from ml.src.data.split import ArchitectureDataSplitter

splitter = ArchitectureDataSplitter(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42)
train_df, val_df, test_df = splitter.group_split_by_architecture(df, group_col="architecture_id")

# Verify zero architecture contamination across splits
train_archs = set(train_df['architecture_id'])
val_archs = set(val_df['architecture_id'])
test_archs = set(test_df['architecture_id'])
overlap_train_test = train_archs.intersection(test_archs)
overlap_train_val = train_archs.intersection(val_archs)

print(f"Train Set Shape:      {train_df.shape} ({len(train_archs)} unique architectures)")
print(f"Validation Set Shape: {val_df.shape} ({len(val_archs)} unique architectures)")
print(f"Test Set Shape:        {test_df.shape} ({len(test_archs)} unique architectures)")
print(f"Architecture ID Overlap between Train and Test: {len(overlap_train_test)}")
assert len(overlap_train_test) == 0, "Leakage Alert: Architecture ID found in both train and test!"
assert len(overlap_train_val) == 0, "Leakage Alert: Architecture ID found in both train and val!"
print("Data splitting methodology passes GroupShuffleSplit leakage audit.")


---
## 8. Summary & Next Steps for Phase 4

### Phase 3 Achievements Documented in this Notebook:
1. **Source Landscape Audited:** 15 data sources cataloged across 5 evidence types in `ml/configs/data.yaml`.
2. **Schema Integrity:** 5 linked datasets (A–E) verified with relational primary/foreign keys.
3. **Fidelity Tracking:** Provenance schema enforced to separate empirical telemetry from synthetic simulations.
4. **Behavioral Dynamics Explored:** Quantified the queuing saturation knee, $p_{99}$ tail latency magnification under service fan-out, and cache alleviation on database workloads.
5. **Graph Metric Pipeline:** Integrated `networkx` topological feature extraction (`graph_depth`, `fan-in`, `fan-out`, `complexity_score`).
6. **Leakage & Quality Handshake:** Validated physical sanity bounds and verified `GroupShuffleSplit` by architecture ID.

### Phase 4 Roadmap:
- Transition to `ml/notebooks/02_data_quality.ipynb` for automated data profiling (missing value audits, z-score outlier detection, drift tests).
- Deepen collaboration with **Ananya** (ML Reviewer) to sign off on formal data quality and leakage prevention gates before surrogate model training.
